# BioJEPA v0.6 Data Prep - Notebook 1: Genes and Splits

This notebook handles:
1. Loading GEARS k562e for official train/val/test perturbation splits
2. Defining cumulative split strategy for all datasets
3. Collecting gene universe from all 6 datasets
4. Selecting final 16,384 genes (k562e-first priority)
5. Generating per-dataset gene masks

**Outputs:**
- `gene_names.json` - ordered list of gene symbols
- `gene_to_idx.json` - ENSG -> index mapping
- `holdout_perturbations.json` - test split perturbations
- `dataset_gene_masks.json` - per-dataset boolean masks
- `cell_type_to_id.json` - cell type name to ID mapping
- `dataset_splits.json` - train/val/test perturbation sets per dataset

In [ ]:
from pathlib import Path
from gears import PertData
from collections import defaultdict
from scipy.sparse import issparse
import pandas as pd
import numpy as np
import scanpy as sc
import json
import gc

In [ ]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'K562_gwps_raw_singlecell_01.h5ad',
    'adamson': ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad',
    'norman': ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad',
    'sciplex': ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad',
}

N_GENES = 16384
MIN_CELLS = 10
CHUNK_SIZE = 50000
VAL_PCT = 0.10
TEST_PCT = 0.05
SEED = 42
np.random.seed(SEED)

## Step 1: Load GEARS k562e Splits

GEARS provides official train/val/test splits for the k562 essential dataset. These are authoritative and will form the basis for all other dataset splits.

In [ ]:
def clean_gears_name(name):
    if name.endswith('+ctrl'):
        return name.replace('+ctrl', '')
    if name.startswith('ctrl+'):
        return name.replace('ctrl+', '')
    if name == 'ctrl':
        return 'control'
    return name.strip()

In [ ]:
pert_data = PertData(ref_dir / 'k562e')
pert_data.load(data_name='replogle_k562_essential')
pert_data.prepare_split(split='simulation', seed=1)

In [ ]:
k562e_train_perts = {clean_gears_name(p) for p in pert_data.set2conditions['train']}
k562e_val_perts = {clean_gears_name(p) for p in pert_data.set2conditions['val']}
k562e_test_perts = {clean_gears_name(p) for p in pert_data.set2conditions['test']}

print(f'k562e splits - train: {len(k562e_train_perts)} | val: {len(k562e_val_perts)} | test: {len(k562e_test_perts)}')

In [ ]:
cumulative_train = set(k562e_train_perts)
cumulative_val = set(k562e_val_perts)
cumulative_test = set(k562e_test_perts)

## Step 2: Cumulative Split Strategy

For each additional dataset, we:
1. Check if perturbations already exist in k562e splits (reuse those assignments)
2. For new perturbations, add holdouts only if current percentage is below target

In [ ]:
def get_dataset_splits(ds_perts, cumulative_train, cumulative_val, cumulative_test, ds_name):
    ds_perts = set(ds_perts)
    n_total = len(ds_perts)
    if n_total == 0:
        return set(), set(), set()

    already_test = ds_perts & cumulative_test
    already_val = ds_perts & cumulative_val
    already_train = ds_perts & cumulative_train
    unassigned = list(ds_perts - cumulative_train - cumulative_val - cumulative_test)
    np.random.shuffle(unassigned)

    current_test_pct = len(already_test) / n_total
    current_val_pct = len(already_val) / n_total

    target_test = int(n_total * TEST_PCT)
    target_val = int(n_total * VAL_PCT)

    needed_test = max(0, target_test - len(already_test))
    needed_val = max(0, target_val - len(already_val))

    new_test = set(unassigned[:needed_test])
    new_val = set(unassigned[needed_test:needed_test + needed_val])
    new_train = set(unassigned[needed_test + needed_val:])

    cumulative_test.update(new_test)
    cumulative_val.update(new_val)
    cumulative_train.update(new_train)

    ds_train = already_train | new_train
    ds_val = already_val | new_val
    ds_test = already_test | new_test

    print(f'{ds_name}: Total {n_total} | Already test: {len(already_test)} ({current_test_pct:.1%}) | '
          f'Added test: {len(new_test)} | Final test: {len(ds_test)} ({len(ds_test)/n_total:.1%})')

    return ds_train, ds_val, ds_test

## Step 3: Collect Gene Universe

For k562e_raw: take ALL genes (no filter)
For other datasets: filter to genes with ncells >= 10

In [ ]:
def count_cells_per_gene_chunked(adata, chunk_size=50000):
    n_cells = adata.n_obs
    n_genes = adata.n_vars
    gene_cell_counts = np.zeros(n_genes, dtype=np.int64)

    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        chunk = adata.X[start:end]
        if issparse(chunk):
            chunk = chunk.toarray()
        gene_cell_counts += (chunk > 0).sum(axis=0).astype(np.int64)
        del chunk
        gc.collect()
        if start % (chunk_size * 10) == 0 and start > 0:
            print(f'  Processed {start:,} / {n_cells:,} cells')

    return gene_cell_counts

In [ ]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True

### k562e_raw (primary - take all genes)

In [ ]:
ds_adata = sc.read_h5ad(datasets['k562e_raw'], backed='r')
var_df = ds_adata.var
k562e_genes = {idx: row['gene_name'] for idx, row in var_df.iterrows() if is_valid(idx)}
print(f'k562e_raw: {len(k562e_genes)} genes (all, no filter)')

In [ ]:
dataset_genes = {'k562e_raw': set(k562e_genes.keys())}
all_genes = dict(k562e_genes)

### rep1e (filter ncells >= 10)

In [ ]:
print('Loading rep1e...')
ds_adata = sc.read_h5ad(datasets['rep1e'], backed='r')
var_df = ds_adata.var.copy()

if 'ncells' in var_df.columns:
    gene_counts = var_df['ncells'].values
else:
    print('Computing cell counts for rep1e...')
    gene_counts = count_cells_per_gene_chunked(ds_adata, CHUNK_SIZE)

valid_mask = gene_counts >= MIN_CELLS
var_df = var_df[valid_mask]
rep1e_genes = {idx: row['gene_name'] for idx, row in var_df.iterrows() if is_valid(idx)}
dataset_genes['rep1e'] = set(rep1e_genes.keys())

for ensg, name in rep1e_genes.items():
    if ensg not in all_genes:
        all_genes[ensg] = name

print(f'rep1e: {len(rep1e_genes)} genes (filtered ncells >= {MIN_CELLS})')
del ds_adata
gc.collect()

### k562gw (filter ncells >= 10)

In [ ]:
print('Loading k562gw...')
ds_adata = sc.read_h5ad(datasets['k562gw'], backed='r')
var_df = ds_adata.var.copy()

k562gw_genes = {idx: row['gene_name'] for idx, row in var_df.iterrows() if is_valid(idx)}

dataset_genes['k562gw'] = set(k562gw_genes.keys())

for ensg, name in k562gw_genes.items():
    if ensg not in all_genes:
        all_genes[ensg] = name

print(f'k562gw: {len(k562gw_genes)} genes (all, var.index is ENSG)')
del ds_adata
gc.collect()

### adamson (filter ncells >= 10)

In [ ]:
print('Loading adamson...')
ds_adata = sc.read_h5ad(datasets['adamson'], backed='r')
var_df = ds_adata.var.copy()

if 'ncells' in var_df.columns:
    gene_counts = var_df['ncells'].values
    valid_mask = gene_counts >= MIN_CELLS
    var_df = var_df[valid_mask]

adamson_genes = {row['ensembl_id']: row.name for _, row in var_df.iterrows() 
                if is_valid(row.get('ensembl_id'))}

dataset_genes['adamson'] = set(adamson_genes.keys())

for ensg, name in adamson_genes.items():
    if ensg not in all_genes:
        all_genes[ensg] = name

print(f'adamson: {len(adamson_genes)} genes (filtered ncells >= {MIN_CELLS})')
del ds_adata
gc.collect()

### norman (filter ncells >= 10)

In [ ]:
print('Loading norman...')
ds_adata = sc.read_h5ad(datasets['norman'], backed='r')
var_df = ds_adata.var.copy()

if 'ncells' in var_df.columns:
    gene_counts = var_df['ncells'].values
    valid_mask = gene_counts >= MIN_CELLS
    var_df = var_df[valid_mask]

ensg_col = 'ensemble_id' if 'ensemble_id' in var_df.columns else 'ensembl_id'
norman_genes = {row[ensg_col]: row.name for _, row in var_df.iterrows() 
               if is_valid(row.get(ensg_col))}

dataset_genes['norman'] = set(norman_genes.keys())

for ensg, name in norman_genes.items():
    if ensg not in all_genes:
        all_genes[ensg] = name

print(f'norman: {len(norman_genes)} genes (filtered ncells >= {MIN_CELLS})')
del ds_adata
gc.collect()

### sciplex (filter ncells >= 10 if available)

In [ ]:
print('Loading sciplex...')
ds_adata = sc.read_h5ad(datasets['sciplex'], backed='r')
var_df = ds_adata.var.copy()

if 'ncells' in var_df.columns:
    gene_counts = var_df['ncells'].values
    valid_mask = gene_counts >= MIN_CELLS
    var_df = var_df[valid_mask]

sciplex_genes = {row['ensembl_id']: row.name for _, row in var_df.iterrows() 
                if is_valid(row.get('ensembl_id'))}

dataset_genes['sciplex'] = set(sciplex_genes.keys())

for ensg, name in sciplex_genes.items():
    if ensg not in all_genes:
        all_genes[ensg] = name

print(f'sciplex: {len(sciplex_genes)} genes')
del ds_adata
gc.collect()

In [ ]:
print(f'\nTotal unique genes across all datasets: {len(all_genes)}')
for ds, genes in dataset_genes.items():
    print(f'  {ds}: {len(genes)}')

## Step 4: Select Final 16,384 Genes

Strategy: k562e-first, then filtered genes from other datasets, preferring genes in multiple datasets

In [ ]:
selected = list(k562e_genes.keys())
print(f'Starting with {len(selected)} genes from k562e')

filtered_genes = {}
for ds_name, ds_genes in dataset_genes.items():
    if ds_name == 'k562e_raw':
        continue
    for g in ds_genes:
        if g not in selected:
            if g not in filtered_genes:
                filtered_genes[g] = []
            filtered_genes[g].append(ds_name)

sorted_genes = sorted(filtered_genes.keys(), key=lambda g: -len(filtered_genes[g]))

for g in sorted_genes:
    if g not in selected:
        selected.append(g)
    if len(selected) >= N_GENES:
        break

print(f'Selected {len(selected)} genes: {len(k562e_genes)} from k562e + {len(selected) - len(k562e_genes)} from other datasets')

In [ ]:
gene_to_idx = {g: i for i, g in enumerate(selected)}
gene_names = [all_genes.get(g, g) for g in selected]

## Step 5: Generate Per-Dataset Gene Masks

In [ ]:
dataset_masks = {}
for ds_name, ds_genes in dataset_genes.items():
    mask = np.zeros(len(selected), dtype=bool)
    for g in ds_genes:
        if g in gene_to_idx:
            mask[gene_to_idx[g]] = True
    dataset_masks[ds_name] = mask.tolist()
    print(f'{ds_name}: {mask.sum()} / {len(selected)} genes covered')

## Step 6: Extract Dataset Perturbations and Assign Splits

In [ ]:
dataset_splits = {
    'k562e_raw': {
        'train': list(k562e_train_perts),
        'val': list(k562e_val_perts),
        'test': list(k562e_test_perts)
    }
}

In [ ]:
def extract_perturbations(ds_key, condition_col='condition'):
    ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
    if condition_col in ds_adata.obs.columns:
        perts = ds_adata.obs[condition_col].unique().tolist()
    elif 'perturbation' in ds_adata.obs.columns:
        perts = ds_adata.obs['perturbation'].unique().tolist()
    elif 'gene' in ds_adata.obs.columns:
        perts = ds_adata.obs['gene'].unique().tolist()
    else:
        print(f'Warning: No perturbation column found in {ds_key}')
        perts = []
    
    ctrl_keywords = ['control', 'ctrl', 'non-targeting', 'vehicle', 'dmso']
    cleaned = []
    for p in perts:
        if is_valid(p):
            cp = clean_gears_name(str(p))
            if cp != 'control' and str(p).lower() not in ctrl_keywords:
                cleaned.append(cp)
    return set(cleaned)

In [ ]:
for ds_key in ['rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']:
    print(f'\nProcessing {ds_key}...')
    perts = extract_perturbations(ds_key)
    print(f'  Found {len(perts)} perturbations')
    
    train, val, test = get_dataset_splits(perts, cumulative_train, cumulative_val, cumulative_test, ds_key)
    
    dataset_splits[ds_key] = {
        'train': list(train),
        'val': list(val),
        'test': list(test)
    }

## Step 7: Save Outputs

In [ ]:
cell_type_to_id = {
    'K562': 0,
    'RPE1': 1,
    'A549': 2,
    'MCF7': 3,
    'unknown': 4
}

In [ ]:
with open(data_dir / 'gene_names.json', 'w') as f:
    json.dump(gene_names, f)

with open(data_dir / 'gene_to_idx.json', 'w') as f:
    json.dump(gene_to_idx, f)

with open(data_dir / 'holdout_perturbations.json', 'w') as f:
    json.dump(list(cumulative_test), f)

with open(data_dir / 'dataset_gene_masks.json', 'w') as f:
    json.dump(dataset_masks, f)

with open(data_dir / 'cell_type_to_id.json', 'w') as f:
    json.dump(cell_type_to_id, f)

with open(data_dir / 'dataset_splits.json', 'w') as f:
    json.dump(dataset_splits, f)

print('Saved outputs to', data_dir)

## Summary

In [ ]:
print('=== Data Prep Notebook 1 Complete ===')
print(f'Selected genes: {len(selected)}')
print(f'Total holdout perturbations: {len(cumulative_test)}')
print(f'Total val perturbations: {len(cumulative_val)}')
print(f'Total train perturbations: {len(cumulative_train)}')
print('\nPer-dataset splits:')
for ds, splits in dataset_splits.items():
    print(f"  {ds}: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")